#대본 데이터(3,000개) 전처리

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
path="/content/drive/MyDrive/mulcam_bigdata/3조 : 최종 프로젝트/공용/csv/대화문.csv"
# path="/content/drive/MyDrive/3조 : 최종 프로젝트/공용/csv/대화문.csv"

In [ ]:
pd.read_csv(path)

,대화,도메인,세부도메인,의도
0,주차 요금 계산할게.,의료,주차 정산,주차비 정산
1,곱창은 돼지를 쓰나 소를 쓰나?,식당/카페,음식 주문/계산,메뉴 정보
2,에스콰이어 최신 호 좀 찾아 주라.,도서관,도서 검색/대출,연속간행물 검색
3,영화 시작 전에 대기할 만한 곳 있어?,영화관,시설 안내,대기 장소
4,이 집에서 제일 인기 있는 메뉴 알려 줘.,식당/카페,음식 주문/계산,메뉴 정보
...,...,...,...,...
2995,장애인 등록 서류 접수하려는데.,복지센터,사회복지,장애인 등록
2996,수강료 수납 받아 주세요.,복지센터,문화센터,수강료 납부
2997,오류 해결하려면 직원 불러야겠지?,지하철,직원 호출,직원 호출
2998,성수행 열차 출발 시간 표시하게.,지하철,배차 시간,출발 시간 검색


In [ ]:
df = pd.read_csv(path)
df.head()

,대화,도메인,세부도메인,의도
0,주차 요금 계산할게.,의료,주차 정산,주차비 정산
1,곱창은 돼지를 쓰나 소를 쓰나?,식당/카페,음식 주문/계산,메뉴 정보
2,에스콰이어 최신 호 좀 찾아 주라.,도서관,도서 검색/대출,연속간행물 검색
3,영화 시작 전에 대기할 만한 곳 있어?,영화관,시설 안내,대기 장소
4,이 집에서 제일 인기 있는 메뉴 알려 줘.,식당/카페,음식 주문/계산,메뉴 정보


In [ ]:
col = ["대화", "도메인"]
df_col = df[col]
df_col.head()

,대화,도메인
0,주차 요금 계산할게.,의료
1,곱창은 돼지를 쓰나 소를 쓰나?,식당/카페
2,에스콰이어 최신 호 좀 찾아 주라.,도서관
3,영화 시작 전에 대기할 만한 곳 있어?,영화관
4,이 집에서 제일 인기 있는 메뉴 알려 줘.,식당/카페


In [ ]:
df_col.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   대화      3000 non-null   object
 1   도메인     3000 non-null   object
dtypes: object(2)
memory usage: 47.0+ KB


In [ ]:
df_col["도메인"].value_counts()

,count
도메인,
도서관,750
지하철,641
의료,475
식당/카페,420
영화관,403
복지센터,311


# KoBERT Model

In [ ]:
!pip install -q torch transformers tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 28.2 MB/s eta 0:00:00


In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("monologg/kobert")
model = AutoModel.from_pretrained("monologg/kobert")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/426 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/77.8k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import re

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("monologg/kobert")

In [ ]:
# PyTorch의 Dataset 클래스를 상속받는 사용자 정의 데이터셋 클래스
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        # 초기화 메서드: 텍스트, 레이블, 토크나이저, 최대 길이를 입력받음
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        # 데이터셋의 길이(샘플 수)를 반환
        return len(self.texts)

    def __getitem__(self, item):
        # 인덱스에 해당하는 샘플을 반환
        text = str(self.texts[item])
        label = self.labels[item]

        # 토크나이저를 사용하여 텍스트를 인코딩
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,  # [CLS], [SEP] 같은 특수 토큰 추가
            max_length=self.max_len,  # 최대 길이 설정
            return_token_type_ids=False,  # token_type_ids 반환하지 않음
            padding='max_length',  # 최대 길이에 맞춰 패딩
            truncation=True,  # 최대 길이를 초과하는 경우 자르기
            return_attention_mask=True,  # attention mask 반환
            return_tensors='pt',  # PyTorch 텐서 형태로 반환
        )

        # 모델 입력에 필요한 형태로 데이터 반환
        return {
            'input_ids': encoding['input_ids'].flatten(),  # 1차원 텐서로 변환
            'attention_mask': encoding['attention_mask'].flatten(),  # 1차원 텐서로 변환
            'labels': torch.tensor(label, dtype=torch.long)  # 레이블을 long 타입 텐서로 변환
        }

In [ ]:
def clean_text(text):
    # 숫자와 '**' 제거
    text = re.sub(r'\d+\*\*', '', text)
    # 맨 앞의 숫자 제거 (있는 경우)
    text = re.sub(r'^\d+', '', text)
    return text.strip()

In [ ]:
# 데이터 준비
texts = df_col["대화"].apply(clean_text).tolist()
label_map = {"도서관": 0, "지하철": 1, "의료": 2, "식당/카페": 3, "영화관": 4, "복지센터": 5}
labels = df_col["도메인"].map(label_map).tolist()

# 데이터 분할
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)

# 데이터셋 생성
train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer, max_len=128)
val_dataset = TextClassificationDataset(val_texts, val_labels, tokenizer, max_len=128)

# DataLoader 생성
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [ ]:
def train_epoch(model, data_loader, optimizer, device):
    model.train()  # 모델을 학습 모드로 설정
    losses = []
    for batch in data_loader:
        optimizer.zero_grad()  # 그래디언트 초기화
        input_ids = batch['input_ids'].to(device)  # 입력 ID를 지정된 디바이스로 이동
        attention_mask = batch['attention_mask'].to(device)  # 어텐션 마스크를 지정된 디바이스로 이동
        labels = batch['labels'].to(device)  # 레이블을 지정된 디바이스로 이동
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)  # 모델에 입력 전달
        loss = outputs.loss  # 손실 계산
        losses.append(loss.item())  # 손실값을 리스트에 추가
        loss.backward()  # 역전파 수행
        optimizer.step()  # 옵티마이저를 사용해 가중치 업데이트
    return np.mean(losses)  # 에폭의 평균 손실 반환

def eval_model(model, data_loader, device):
    model.eval()  # 모델을 평가 모드로 설정
    predictions = []
    actual_labels = []
    with torch.no_grad():  # 그래디언트 계산 비활성화
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)  # 입력 ID를 지정된 디바이스로 이동
            attention_mask = batch['attention_mask'].to(device)  # 어텐션 마스크를 지정된 디바이스로 이동
            labels = batch['labels'].to(device)  # 레이블을 지정된 디바이스로 이동
            outputs = model(input_ids, attention_mask=attention_mask)  # 모델에 입력 전달
            _, preds = torch.max(outputs.logits, dim=1)  # 가장 높은 확률의 클래스 선택
            predictions.extend(preds.cpu().tolist())  # 예측값을 CPU로 이동 후 리스트에 추가
            actual_labels.extend(labels.cpu().tolist())  # 실제 레이블을 CPU로 이동 후 리스트에 추가
    accuracy = accuracy_score(actual_labels, predictions)  # 정확도 계산
    # 분류 보고서 생성 및 정확도와 함께 반환
    return classification_report(actual_labels, predictions, target_names=label_map.keys()), accuracy

In [ ]:
# 모델 초기화
model = AutoModelForSequenceClassification.from_pretrained("monologg/kobert", num_labels=len(label_map))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at monologg/kobert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# 학습
epochs = 100
patience = 5  # 조기 종료를 위한 인내심
best_accuracy = 0
no_improvement = 0

for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, optimizer, device)
    report, accuracy = eval_model(model, val_loader, device)
    print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Validation Accuracy: {accuracy:.4f}')
    print(report)

    # 조기 종료 로직
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        no_improvement = 0
        # 최고 성능 모델 저장
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        no_improvement += 1

    if no_improvement >= patience:
        print(f"Early stopping triggered. No improvement for {patience} epochs.")
        break

# 학습 완료 후 최고 성능 모델 로드
model.load_state_dict(torch.load('best_model.pth'))
print(f"Best validation accuracy: {best_accuracy:.4f}")

Epoch 1/100, Train Loss: 1.0526, Validation Accuracy: 0.5100
              precision    recall  f1-score   support

         도서관       0.56      0.63      0.59       150
         지하철       0.51      0.60      0.55       128
          의료       0.43      0.48      0.46        95
       식당/카페       0.51      0.55      0.53        84
         영화관       0.64      0.22      0.33        81
        복지센터       0.44      0.39      0.41        62

    accuracy                           0.51       600
   macro avg       0.52      0.48      0.48       600
weighted avg       0.52      0.51      0.50       600

Epoch 2/100, Train Loss: 1.0012, Validation Accuracy: 0.4867
              precision    recall  f1-score   support

         도서관       0.58      0.53      0.55       150
         지하철       0.43      0.65      0.52       128
          의료       0.48      0.41      0.44        95
       식당/카페       0.48      0.55      0.51        84
         영화관       0.59      0.23      0.34        81
        복지